In [ ]:
!pip -q install openai pandas
import pandas as pd, json, time, os, glob, html as h
from openai import OpenAI
from google.colab import userdata, output
from IPython.display import HTML, display
output.no_vertical_scroll()

# ---------- Setup ----------
KEY = userdata.get("GROQ_API_KEY").strip()
assert KEY.startswith("gsk_"), f"Ye Groq key nahi hai (shuru: {KEY[:4]})"
client = OpenAI(api_key=KEY, base_url="https://api.groq.com/openai/v1")
avail = [m.id for m in client.models.list().data]
MODEL = next(m for m in ["openai/gpt-oss-120b","llama-3.3-70b-versatile","llama-3.1-8b-instant","openai/gpt-oss-20b"] if m in avail)

files = glob.glob("Chinar_Quantum*.csv")
assert files, "CSV Colab Files me nahi mili. Pehle upload karo."
CSV = max(files, key=os.path.getmtime)          # sabse nayi CSV
CACHE = "students_v3.json"
print("CSV:", CSV, "| Model:", MODEL)

# ---------- Grok/Groq se extraction ----------
PROMPT = """You prepare a professional learner profile for an organization's management.
Input: one learner's feedback row. Return ONLY JSON:
{"name": str, "education": short badge text or "",
 "org_feedback": 1-2 sentences on what the learner said about the organization/program,
 "goals": [learner's stated goals/aspirations],
 "status": [current job/study status, background, skills the learner stated about themselves],
 "org_value": [3-4 insights useful for the organization: strengths, commitment, engagement potential],
 "improvements": [2-4 areas where this learner needs development or support, based on gaps/challenges in the data]}
RULES:
- Professional third person ("The learner ..."), concise, no slang.
- name, education, org_feedback, goals, status, org_value: ONLY facts clearly stated about THIS learner.
- NEVER attribute to the learner anything said about mentors, staff or other people.
- Family, marital or personal details: include ONLY if the learner explicitly states them. If in doubt, omit.
- If nothing was said for a field, return empty list or "". Never invent facts. Each item under 22 words."""

def extract(row):
    text = "\n".join(f"{k}: {v}" for k, v in row.items() if pd.notna(v))
    for t in range(3):
        try:
            r = client.chat.completions.create(model=MODEL, temperature=0, response_format={"type":"json_object"},
                messages=[{"role":"system","content":PROMPT},{"role":"user","content":text}])
            return json.loads(r.choices[0].message.content)
        except Exception as ex:
            if t == 2: raise
            time.sleep(6)

def load_data(refresh=False):
    if os.path.exists(CACHE) and not refresh:
        return json.load(open(CACHE, encoding="utf-8"))
    df = pd.read_csv(CSV)
    df = df[df["Role"].astype(str).str.strip().str.lower() == "learner"]   # mentors skip
    data = []
    for i, row in df.iterrows():
        try: data.append(extract(row)); time.sleep(1.5)
        except Exception as ex: print(f"Row {i} failed: {ex}"); break
    json.dump(data, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    return data

# ---------- Card design ----------
BAD = ("married", "daughter", "family", "wife", "husband", "children")
PAL = [("#4f6fe8","#eef2ff"), ("#2fa37f","#ecfaf4"), ("#e5607e","#fff0f4"), ("#e0912b","#fff7e8"),
       ("#3b9ad9","#eef8ff"), ("#7d6be6","#f4f1ff"), ("#d9695f","#fff2f0"), ("#1fa5a5","#eafafa")]
SECTIONS = [("goals","🎯","Goals & Aspirations"), ("status","💼","Current Status & Background"),
            ("org_value","📊","Key Insights for Organization"), ("improvements","🛠️","Improvement Areas & Support Needed")]

def card(d, n):
    e = h.escape
    a, s = PAL[(n-1) % len(PAL)]
    name = d.get("name") or "Learner"
    edu = f"<span class='badge'>🎓 {e(d['education'])}</span>" if d.get("education") else ""
    org = (f"<div class='org'><div class='ol'>💬 ORGANIZATION FEEDBACK / ALIGNMENT</div><p>{e(d['org_feedback'])}</p></div>"
           if d.get("org_feedback") else "")
    secs = ""
    for key, ico, title in SECTIONS:
        items = [x for x in (d.get(key) or []) if not any(b in str(x).lower() for b in BAD)]
        if items:
            lis = "".join(f"<li>{e(str(i))}</li>" for i in items)
            secs += f"<div class='sec'><div class='st'><span>{ico}</span>{title}</div><ul>{lis}</ul></div>"
    return f"""<div class='card' style='--a:{a};--s:{s}'>
      <div class='head'><div class='av'>{e(name.strip()[:1].upper())}</div>
        <div class='nm'><h2>{e(name)}</h2>{edu}</div><div class='no'>#{n}</div></div>
      <div class='body'>{org}<div class='cols'>{secs}</div></div></div>"""

CSS = """<style>
.wrap{font-family:'Segoe UI',Roboto,sans-serif;background:#f4f6fb;padding:28px 22px;border-radius:16px;box-sizing:border-box;width:100%}
.title{text-align:center;font-size:34px;font-weight:800;color:#2b3a67;margin:0}
.sub{text-align:center;color:#5b6475;margin:6px 0 28px;font-size:17px;font-weight:600}
.list{width:100%;display:flex;flex-direction:column;gap:32px}
.card{background:#fff;border-radius:22px;overflow:hidden;box-shadow:0 8px 26px #2b3a6720;border:2px solid var(--s)}
.head{display:flex;align-items:center;gap:22px;padding:26px 34px;background:var(--a)}
.av{width:78px;height:78px;border-radius:50%;background:#fff;color:var(--a);font-size:38px;font-weight:800;display:flex;align-items:center;justify-content:center;flex:none}
.nm{flex:1}.nm h2{margin:0 0 10px;color:#fff;font-size:32px;font-weight:800}
.badge{display:inline-block;background:#ffffff33;color:#fff;border:1.5px solid #ffffffaa;padding:7px 18px;border-radius:30px;font-size:16px;font-weight:700}
.no{color:#ffffffcc;font-size:26px;font-weight:800}
.body{padding:26px 34px 32px;background:#fff}
.org{background:var(--s);border:2px solid var(--a);border-left:10px solid var(--a);border-radius:16px;padding:20px 26px;margin-bottom:22px}
.ol{font-size:16px;font-weight:800;letter-spacing:1px;color:var(--a);margin-bottom:8px}
.org p{margin:0;font-size:21px;line-height:1.55;font-weight:700;color:#1f2937}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:20px}
.sec{background:var(--s);border-left:6px solid var(--a);border-radius:14px;padding:18px 24px}
.st{font-size:21px;font-weight:800;margin-bottom:10px;display:flex;align-items:center;gap:10px;color:var(--a)}
.st span{font-size:24px}
.sec ul{margin:0;padding-left:22px;font-size:18px;line-height:1.65;font-weight:600;color:#1f2937}
.sec li{margin:6px 0}
@media(max-width:850px){.cols{grid-template-columns:1fr}.head{padding:20px}.body{padding:20px}}
@media print{.card{break-inside:avoid;box-shadow:none}}
</style>"""

# ---------- Run ----------
def build_profiles(refresh=False, out="student_cards.html"):
    data = load_data(refresh)
    page = (f"{CSS}<div class='wrap'><div class='title'>Learner Profile Report</div>"
            f"<div class='sub'>{len(data)} learners • Chinar Quantum AI Learners</div>"
            f"<div class='list'>{''.join(card(d, i+1) for i, d in enumerate(data))}</div></div>")
    open(out, "w", encoding="utf-8").write(page)
    display(HTML(page)); print(f"✅ {len(data)} learner cards → {out}")

build_profiles()

<IPython.core.display.Javascript object>

CSV: Chinar_Quantum_AI_Learners_Feedback_Profile (1).csv | Model: openai/gpt-oss-120b


✅ 19 learner cards → student_cards.html
